<!-- Notebook purpose: clone the project in Kaggle, build/run its Bash workflow, and analyze the experiment outputs. -->
# Offline 1-CTP experiments on Kaggle

Set `REPO_URL` in the configuration cell. For a private GitHub repository, add a Kaggle Secret named `GITHUB_TOKEN` with a read-only repository token; do not paste the token into this notebook or its outputs. The notebook reads that secret through Kaggle's secret store and does not put credentials in the clone URL. For a public repository, set `GITHUB_TOKEN_SECRET_NAME = None`.

Run the cells in order. The workflow clones or fast-forward updates the repository, builds and tests the C++ project, invokes `scripts/run_experiments.sh`, renders DOT files with `scripts/render_graphs.sh`, and analyzes the results. Edit the repository's `config/experiment_config.csv` to choose graph sizes, seeds, and repetitions before the run.

In [ ]:
import base64
import importlib.util
import os
import shutil
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlparse

# Replace this URL with the repository you want to run.
REPO_URL = "https://github.com/hcuni2022-ctrl/comp3821_prj.git"
GITHUB_TOKEN_SECRET_NAME = "GITHUB_TOKEN"  # Set to None for a public repository.

PROJECT_ROOT = Path("/kaggle/working/comp3821_prj")
EXPERIMENT_DIR = PROJECT_ROOT / "ctp-experiment"
RESULTS_PATH = EXPERIMENT_DIR / "data/results/results.csv"
IMAGE_DIR = EXPERIMENT_DIR / "visualisations/images"

parsed_repo = urlparse(REPO_URL)
if parsed_repo.scheme != "https" or not parsed_repo.hostname:
    raise ValueError("REPO_URL must be an HTTPS repository URL")
if parsed_repo.username or parsed_repo.password:
    raise ValueError("Do not embed credentials in REPO_URL; use Kaggle Secrets")
print(f"Repository: {REPO_URL}")
print(f"Experiment directory: {EXPERIMENT_DIR}")

In [ ]:
git_env = os.environ.copy()
if GITHUB_TOKEN_SECRET_NAME is not None:
    if parsed_repo.hostname.lower() != "github.com":
        raise ValueError("The configured token flow is limited to github.com URLs")
    from kaggle_secrets import UserSecretsClient

    github_token = UserSecretsClient().get_secret(GITHUB_TOKEN_SECRET_NAME)
    if not github_token:
        raise ValueError(f"Kaggle Secret {GITHUB_TOKEN_SECRET_NAME!r} is empty")
    auth = base64.b64encode(f"x-access-token:{github_token}".encode()).decode()
    git_env.update({
        "GIT_CONFIG_COUNT": "1",
        "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
        "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}",
    })
    del github_token, auth

try:
    if PROJECT_ROOT.exists():
        if not (PROJECT_ROOT / ".git").is_dir():
            raise FileExistsError(
                f"{PROJECT_ROOT} exists but is not a Git clone; choose another PROJECT_ROOT"
            )
        remote = subprocess.run(
            ["git", "-C", str(PROJECT_ROOT), "remote", "get-url", "origin"],
            check=True, capture_output=True, text=True,
        ).stdout.strip()
        if remote != REPO_URL:
            raise ValueError(f"Existing clone remote does not match REPO_URL: {remote}")
        subprocess.run(
            ["git", "-C", str(PROJECT_ROOT), "pull", "--ff-only"],
            check=True, env=git_env,
        )
        print("Updated existing clone with a fast-forward pull.")
    else:
        subprocess.run(
            ["git", "clone", "--depth", "1", REPO_URL, str(PROJECT_ROOT)],
            check=True, env=git_env, capture_output=True, text=True,
        )
        print("Repository cloned.")
finally:
    for key in ("GIT_CONFIG_COUNT", "GIT_CONFIG_KEY_0", "GIT_CONFIG_VALUE_0"):
        git_env.pop(key, None)
    del git_env

if not (EXPERIMENT_DIR / "scripts/run_experiments.sh").is_file():
    raise FileNotFoundError(f"Expected experiment project at {EXPERIMENT_DIR}")

In [ ]:
# Install only the Kaggle runtime dependencies that are missing.
apt_packages = []
if shutil.which("g++") is None or shutil.which("make") is None:
    apt_packages.append("build-essential")
if not Path("/usr/include/boost/graph/adjacency_list.hpp").is_file():
    apt_packages.append("libboost-graph-dev")
if shutil.which("dot") is None:
    apt_packages.append("graphviz")
if apt_packages:
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", *apt_packages], check=True)

python_packages = [
    package for module, package in (("pandas", "pandas"), ("matplotlib", "matplotlib"))
    if importlib.util.find_spec(module) is None
]
if python_packages:
    subprocess.run([sys.executable, "-m", "pip", "install", *python_packages], check=True)

# Use the repository's build script, then run its focused C++ tests.
subprocess.run(["bash", "scripts/build.sh"], cwd=EXPERIMENT_DIR, check=True)
subprocess.run(["make", "test"], cwd=EXPERIMENT_DIR, check=True)

In [ ]:
config_path = EXPERIMENT_DIR / "config/experiment_config.csv"
if not config_path.is_file():
    raise FileNotFoundError(f"Experiment configuration not found: {config_path}")
print(config_path.read_text())

# This script builds as needed and runs the configured batch.
subprocess.run(["bash", "scripts/run_experiments.sh"], cwd=EXPERIMENT_DIR, check=True)
if not RESULTS_PATH.is_file():
    raise FileNotFoundError(f"Experiment results were not created: {RESULTS_PATH}")

In [ ]:
# Convert each saved DOT graph to a PNG using the project rendering script.
subprocess.run(["bash", "scripts/render_graphs.sh"], cwd=EXPERIMENT_DIR, check=True)
print(f"Rendered graph images: {IMAGE_DIR}")

In [ ]:
import json

import matplotlib.pyplot as plt
import pandas as pd

OUTPUT_DIR = EXPERIMENT_DIR / "data/results/analysis"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
results = pd.read_csv(RESULTS_PATH)

required_columns = {
    "status", "vertex_count", "edge_count", "edge_seed", "blockage_seed",
    "initial_path_cost", "algorithm_paths", "scenario_metrics", "preprocessing_us",
    "recovery_lookup_us", "actual_recovery_cost", "actual_total_travel_cost",
    "worst_case_total_travel_cost", "worst_case_competitive_ratio",
}
missing = required_columns.difference(results.columns)
if missing:
    raise ValueError(f"Results CSV is missing columns: {sorted(missing)}")

successful = results.loc[results["status"] == "ok"].copy()
infeasible = results.loc[results["status"] == "infeasible"].copy()
print(f"Successful runs: {len(successful)}; infeasible plans: {len(infeasible)}")
results.head()

In [ ]:
scenario_records = []
for row in successful.itertuples():
    paths = json.loads(row.algorithm_paths)
    if not paths or any(not path for path in paths):
        raise ValueError(f"Run {row.run_id} has an empty algorithm path")
    if paths[-1] != [int(row.destination)]:
        raise ValueError(f"Run {row.run_id} is missing the terminal destination entry")
    initial_path = [path[0] for path in paths]
    if initial_path[0] != int(row.source) or initial_path[-1] != int(row.destination):
        raise ValueError(f"Run {row.run_id} has an invalid initial-path projection")
    for scenario in json.loads(row.scenario_metrics):
        scenario_records.append({"run_id": row.run_id, "vertex_count": row.vertex_count, **scenario})

scenario_results = pd.DataFrame(scenario_records)
if not scenario_results.empty:
    scenario_summary = scenario_results.groupby("vertex_count").agg(
        scenarios=("path_index", "count"),
        mean_recovery_cost=("recovery_cost", "mean"),
        mean_total_travel_cost=("total_travel_cost", "mean"),
        worst_competitive_ratio=("competitive_ratio", "max"),
    ).reset_index()
    display(scenario_summary)
else:
    print("No feasible recovery scenarios are available to summarize.")

In [ ]:
metrics = [
    "graph_generation_us", "preprocessing_us", "recovery_lookup_us",
    "actual_recovery_cost", "actual_total_travel_cost",
    "worst_case_total_travel_cost", "worst_case_competitive_ratio",
]
summary = results.groupby("vertex_count").agg(
    runs=("run_id", "count"),
    infeasible_runs=("status", lambda status: (status == "infeasible").sum()),
    **{f"mean_{metric}": (metric, "mean") for metric in metrics},
).reset_index()
display(summary)

if successful.empty:
    print("No feasible plans are available to plot yet.")
else:
    successful["vertex_count"] = pd.to_numeric(successful["vertex_count"])
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    for column, label in [
        ("preprocessing_us", "Offline preprocessing (us)"),
        ("worst_case_total_travel_cost", "Worst-case travel cost"),
    ]:
        grouped = successful.groupby("vertex_count")[column].mean()
        axes[0 if column == "preprocessing_us" else 1].plot(
            grouped.index, grouped.values, marker="o", label=label
        )
    axes[0].set(xlabel="Vertices", ylabel="Microseconds", title="Offline preprocessing")
    axes[1].set(xlabel="Vertices", ylabel="Cost", title="Worst-case Backtrack travel")
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "cost_and_preprocessing.png", dpi=160)
    plt.show()

    successful.boxplot(column="worst_case_competitive_ratio", by="vertex_count")
    plt.title("Backtrack worst-case competitive ratio")
    plt.suptitle("")
    plt.xlabel("Vertices")
    plt.ylabel("Ratio vs. offline shortest route")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "competitive_ratio.png", dpi=160)
    plt.show()

In [ ]:
from IPython.display import Image, display

rendered_images = sorted(IMAGE_DIR.glob("*.png"))
if not rendered_images:
    print("No rendered graph images were found.")
else:
    for image_path in rendered_images[:6]:
        print(image_path.name)
        display(Image(filename=str(image_path)))

Interpret preprocessing time separately from recovery lookup time: recovery routes are computed before travel. Infeasible rows indicate that removing at least one initial-path edge disconnects the destination, so no complete all-scenarios plan is emitted for that run. Compare multiple seeds and graph sizes before drawing conclusions; this baseline does not imply the proposed path-tree approach is faster or better.